Primero hacemos una prueba de que la importación de los datos se hace de forma correcta desde el disco HHD `D:` usando las direcciónes del `.env` y de `config.py`, y que las funciones del modulo `loader.py` funcionan correctamente

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import sys
import os
import numpy as np
import pandas as pd

# 1. Cargar .env desde la raíz del proyecto
project_root = Path.cwd().parents[0]
load_dotenv(project_root / ".env")

# 2. Añadir raíz del proyecto al sys.path
sys.path.insert(0, str(project_root))

# 3. Importar loader que importa de config.py correctamente las rutas
from src.data.loader import get_quarters, load_quarter, load_failures

# 4. Verificar
from src.utils.config import DATA_ROOT, RAW_DIR
print("DATA_ROOT:", DATA_ROOT)
print("RAW_DIR:", RAW_DIR)

# 5. Probar carga
quarters = get_quarters()
print(f"Trimestres: {len(quarters)}")

df = load_quarter(quarters[0], "FTS")
print(df.shape)



DATA_ROOT: D:\financial_risk_data
RAW_DIR: D:\financial_risk_data\dataraw
Trimestres: 40
(6193, 3369)


Veamos que estructura tienen los archivos de un trimestre completo, i.e., cuántas filas tiene cada archivo, cuántas columnas, si falta algun archivo.

In [2]:
quarters = get_quarters()

file_types = ["FTS", "CDI", "RAT", "MERG", "STRU"]

shapes = []

for q in quarters:
    suffix = q.name[3:]
    for ft in file_types:
        try:
            df = load_quarter(q, ft)
            shapes.append({
                "quarter": suffix,
                "file": ft,
                "rows": df.shape[0],
                "cols": df.shape[1]
            })
        except FileNotFoundError:
            shapes.append({
                "quarter": suffix,
                "file": ft,
                "rows": None,
                "cols": None
            })

pd.DataFrame(shapes)

,quarter,file,rows,cols
0,1603,FTS,6193,3369
1,1603,CDI,6193,1097
2,1603,RAT,6193,253
3,1603,MERG,26336,61
4,1603,STRU,6546,122
...,...,...,...,...
195,2512,FTS,4408,3369
196,2512,CDI,4408,1097
197,2512,RAT,4408,253
198,2512,MERG,28422,61


El resultado es que tenemos 200 filas (40 trimestres por 5 archivos de datos)

In [3]:
# el número total de filas de todos los trimestres por Base de datos  (FTS, RAT, CDI....)
def total_rows(file_type):
    total = 0
    for q in get_quarters():
        try:
            df = load_quarter(q, file_type)
            total += len(df)
        except FileNotFoundError:
            pass
    return total

totals = {ft: total_rows(ft) for ft in file_types}
totals

{'FTS': 206129, 'CDI': 206129, 'RAT': 206129, 'MERG': 1100520, 'STRU': 219576}

In [4]:
def validate_files():
    quarters = get_quarters()
    missing = []

    for q in quarters:
        suffix = q.name[3:]
        for ft in file_types:
            expected = q / f"{ft}{suffix}.csv"
            if not expected.exists():
                missing.append(str(expected))

    return missing

missing_files = validate_files()
missing_files

[]

In [5]:
def explore_columns():
    sample_quarter = get_quarters()[0]
    info = {}

    for ft in file_types:
        df = load_quarter(sample_quarter, ft)
        info[ft] = list(df.columns)

    return info

columns_info = explore_columns()
columns_info


{'FTS': ['AASLRIND',
  'AATOTFV',
  'ABCUBK',
  'ABCUOTH',
  'ABCXBK',
  'ABCXOTH',
  'ACEPT',
  'ACEPTDOM',
  'ACEPTFOR',
  'ACLPCDA',
  'AGLOSEND',
  'ALLCRCD',
  'ALNSCHTM',
  'ANNUASST',
  'ANNUSALE',
  'APCDLNLS',
  'APCDOFA',
  'APCDSCHM',
  'ASCEAUTO',
  'ASCECI',
  'ASCECONS',
  'ASCECRCD',
  'ASCEHEL',
  'ASCEOTH',
  'ASCERES',
  'ASDRAUTO',
  'ASDRCI',
  'ASDRCONS',
  'ASDRCRCD',
  'ASDRHEL',
  'ASDROTH',
  'ASDRRES',
  'ASDRTL',
  'ASSET',
  'ASSET0',
  'ASSET10',
  'ASSET100',
  'ASSET150',
  'ASSET20',
  'ASSET250',
  'ASSET300',
  'ASSET4',
  'ASSET400',
  'ASSET50',
  'ASSET600',
  'ASSET625',
  'ASSET937',
  'ASSETAG',
  'ASSETCON',
  'ASSETDOM',
  'ASSETELG',
  'ASSETFOR',
  'ASSETIBF',
  'ASSETMNT',
  'ASSETPLG',
  'ASSETRE',
  'ASSETREC',
  'ASSETTFR',
  'ASSETTR',
  'ASSLD100',
  'ASSLDCEA',
  'ASSLDRCR',
  'ASSNETDD',
  'ASST1250',
  'ASST2',
  'ASTCRSUB',
  'ASTNETTM',
  'ASTSERV',
  'ATOTFV',
  'ATOTL1FV',
  'ATOTL2FV',
  'ATOTL3FV',
  'ATOTNTFV',
  'ATRR',
  'AT

Como vemos tenemos un conjunto de columnas intratable a simple vista, unas 4902 columnas repatidas en cinco bases de datos diferentes que componen el dataset de _RSI_. Para saber que son las etiquetas y a que hacen referencia podemos consultar esos datos en la propia base de datos de La FDIC (Federal Deposit Insurance Corporation) en la pagina de _RIS Dictionary_. El RIS no es un solo dataset, sino una familia de bases con funciones distintas dentro del mismo sistema. Las distintas bases de datos son las siguientes:

>`FTS`: Contiene la serie temporal financiera “base”, es decir, datos financieros fuente por trimestre, y es la mejor candidata para construir tu panel temporal principal.

>`CDI`: Agrupa cantidades enteras derivadas matemáticamente a partir de STRU, MERG y FTS; suele servir para variables cuantitativas intermedias o agregadas.

>`RAT`: Reúne ratios financieros derivados de STRU, MERG, FTS y CDI; es muy útil porque traduce estados financieros a indicadores interpretables de riesgo.

>`STRU`: Contiene información estructural y no financiera de la institución, como localización, agencia reguladora, charter y características básicas derivadas de otras fuentes institucionales.

>`MERG`: Este sería el archivo de fusiones/cambios estructurales. Su valor es más event-driven: te ayuda a detectar cambios en la entidad o a explicar discontinuidades en la serie.

Una vez que ya sabemos que valores y datos almacenan cada una de las bases de datos, lo primero que haremos es concatenar todos los trimestres de cada base de datos para asi en vez de tener los archivos de datos por trimestres tenerlos por categoria, creando asi cinco paneles completos que seran las bases de datos a lo largo de todos los trimestres. Estos paneles los guardaremos en el contenedor de datos que hemos creado a parte del repositorio, donde tambien estan los `dataraw`, `D:/financial_risk_data/processed/`. Los almacenaremos en formato _parquet_, ya que ocupan 3-4x menos memoria que CSV y carga mucho más rápido. Cada panel sería un archivo:
```p
processed/
    panel_FTS.parquet
    panel_CDI.parquet
    panel_RAT.parquet
    panel_MERG.parquet
    panel_STRU.parquet
``` 


Una observación importante a tener en cuenta es que los valores numericos en las bases de datos tienen comas como separador de miles y eso puede suponer un problema ya que `Pandas` no puede convertirlos a numérico directamente porque interpreta la coma como string. Luego tenemos que quitar las comas antes de convertir.

In [6]:
import src
from src.utils.config import PROCESSED_DIR

print("PROCESSED_DIR:", PROCESSED_DIR)

def clean_dtypes(df: pd.DataFrame) -> pd.DataFrame: 
    """Limpia tipos mixtos y elimina separadores de miles."""
    for col in df.columns:
        if df[col].dtype == 'object' or str(df[col].dtype) == 'string':
            cleaned = df[col].astype(str).str.replace(',', '', regex=False)
            converted = pd.to_numeric(cleaned, errors='coerce')
            if converted.notna().sum() > len(df) * 0.1:
                df[col] = converted
            else:
                df[col] = df[col].astype(str).replace('nan', pd.NA)
    return df


def build_panel(file_type: str) -> pd.DataFrame:
    quarters = get_quarters()
    frames = []
    quarter_map = {'03': 'Q1', '06': 'Q2', '09': 'Q3', '12': 'Q4'}
    
    for q in quarters:
        try:
            df = load_quarter(q, file_type)
            suffix = q.name[3:]
            year = '20' + suffix[:2]
            df = df.copy()
            df['period'] = year + quarter_map[suffix[2:]]
            df = clean_dtypes(df)
            frames.append(df)
            print(f"  {q.name}: {df.shape}")
        except Exception as e:
            print(f"  Warning {q.name}: {e}")
    
    panel = pd.concat(frames, ignore_index=True)
    
    # Segunda pasada ligera — resolver conflictos de tipo entre trimestres
    for col in panel.columns:
        if str(panel[col].dtype) in ['object', 'string']:
            converted = pd.to_numeric(
                panel[col].astype(str).str.replace(',', '', regex=False), 
                errors='coerce'
            )
            if converted.notna().sum() > len(panel) * 0.05:
                panel[col] = converted
            else:
                # Forzar string puro para parquet
                panel[col] = panel[col].astype(str).where(
                    panel[col].notna(), other=None
                )
    
    print(f"\nPanel {file_type} total: {panel.shape}")
    return panel

# construimos los paneles y los guardamos
for file_type in ['FTS', 'CDI', 'RAT', 'MERG', 'STRU']:
    print(f"\n{'='*40}")
    print(f"Procesando {file_type}...")
    panel = build_panel(file_type)
    output_path = PROCESSED_DIR / f"panel_{file_type}.parquet"
    panel.to_parquet(output_path, index=False)
    print(f"Guardado: {output_path}")

#verificamos que se han creado en la carpeta correcta
for file_type in ['FTS', 'CDI', 'RAT', 'MERG', 'STRU']:
    df = pd.read_parquet(PROCESSED_DIR / f"panel_{file_type}.parquet")
    n_cert = df['CERT'].nunique() if 'CERT' in df.columns else 'N/A'
    print(f"{file_type}: {df.shape} | periods: {df['period'].nunique()} | CERT únicos: {n_cert}")

PROCESSED_DIR: D:\financial_risk_data\processed

Procesando FTS...
  ris1603: (6193, 3370)
  ris1606: (6129, 3370)
  ris1609: (6051, 3370)
  ris1612: (5983, 3370)
  ris1703: (5925, 3370)
  ris1706: (5856, 3370)
  ris1709: (5806, 3370)
  ris1712: (5738, 3370)
  ris1803: (5674, 3370)
  ris1806: (5610, 3370)
  ris1809: (5544, 3370)
  ris1812: (5473, 3370)
  ris1903: (5428, 3370)
  ris1906: (5369, 3370)
  ris1909: (5325, 3370)
  ris1912: (5244, 3370)
  ris2003: (5182, 3370)
  ris2006: (5131, 3370)
  ris2009: (5098, 3370)
  ris2012: (5067, 3370)
  ris2103: (5044, 3370)
  ris2106: (5015, 3370)
  ris2109: (4979, 3370)
  ris2112: (4904, 3370)
  ris2203: (4861, 3370)
  ris2206: (4838, 3370)
  ris2209: (4813, 3370)
  ris2212: (4773, 3370)
  ris2303: (4739, 3370)
  ris2306: (4713, 3370)
  ris2309: (4685, 3370)
  ris2312: (4657, 3370)
  ris2403: (4639, 3370)
  ris2406: (4609, 3370)
  ris2409: (4588, 3370)
  ris2412: (4559, 3370)
  ris2503: (4535, 3370)
  ris2506: (4493, 3370)
  ris2509: (4451, 337

Una vez tenemos los paneles de cada base de datos la idea principal es pensar en estos como diferentes subdataset con los que crearemos los _tabular data_, que seran paneles temporales, para la ingesta del modelo transformer de TabPFN, mediante la union de los paneles (`FTS` + `CDI` + `RAT`) mediante y crear una vista estructural con (`STRU`), la cual define los nodos y atributos del grafo temporal `T_GCN` o `EvolveGCN` y una vista de eventos con (`MERG`), que genera aristas dinámicas y discontinuidades para el grafo.

El problema de intentar hacer esto con los paneles crea problemas ya que si unieramos todas las columnas en un solo dataset tendriamos 4719 columnas, pero `TabPFN-2.5` escala hasta 2,000 features/columnas en modo estándar, luego el modelo estaria entrando en una zona donde no estaría cubriendo el dataset completo de forma directa. Luego necesitaremos hacer ingenieria de datos y de caracteristicas: La selección de columnas relevantes dentro de `FTS`, `CDI`, `RAT`, `STRU` y `MERG` pertenece a la _ingeniería de características_. La construcción del panel temporal, la limpieza, la unificación por `CERT–DATE` y la preparación del grafo dinámico pertenecen a la _ingeniería de datos_.

Como vemos los datos reflejan que `FTS`, `CDI` y `RAT` tienen exactamente 206.129 filas y 6.282 `CERT` únicos en los mismos 40 periodos, es decir, son la misma población de bancos vista desde tres ángulos distintos. El join entre ellas se hace mediante `CERT + period` de forma clara.

`STRU` tiene 6.638 CERT únicos (más que FTS/CDI/RAT). Eso indica que `STRU` incluye entidades que ya no reportan financieramente pero siguen en el registro estructural (bancos cerrados, absorbidos). Aun asi sigue siendo información útil para los nodos del grafo.

`MERG` tiene solo 3.059 `CERT` únicos sobre 1.100.520 filas, lo cual indica muchos eventos por banco. No se une directamente al panel, se agrega primero.

La forma teorica de abordar esta problematica es mediante un preprocesado en capas.

__Capa A — Marco teórico CAMELS__

`CAMELS` es un marco de evaluación bancaria que organiza el análisis del riesgo y la solidez de una entidad en seis dimensiones: Capital adequacy, Asset quality, Management, Earnings, Liquidity y Sensitivity to market risk. El valor de esta evaluación ofrece una estructura reconocible y trazable para seleccionar variables financieras con significado económico y regulatorio. La literatura sobre early warning systems y diagnóstico de fragilidad bancaria ha utilizado de forma recurrente indicadores alineados con `CAMELS` para anticipar deterioro, insolvencia o problemas de supervisión. En ese sentido, mapear variables del dataset a estas dimensiones no es una decisión arbitraria, sino una forma de anclar la ingeniería de variables en un marco interpretativo consolidado.
Usar `CAMELS` permite convertir un conjunto amplio de variables bancarias en bloques conceptuales más manejables. Por ejemplo, ratios de capital se asocian con C, indicadores de morosidad o calidad de activos con A, métricas de eficiencia con M, rentabilidad con E, liquidez con L y exposición a tipos de interés o mercado con S. Esa agrupación facilita tanto la selección de features como la discusión de resultados, porque puedes explicar qué dimensión del perfil bancario está impulsando el riesgo predicho.

En este trabajo adoptamos `CAMELS` como marco teórico para estructurar la selección de variables bancarias. `CAMELS` organiza la evaluación prudencial en seis dimensiones —capital, calidad de activos, gestión, rentabilidad, liquidez y sensibilidad al riesgo de mercado— y ha sido ampliamente utilizado en la literatura de solvencia bancaria y sistemas de alerta temprana.


__Capa B — Filtros estadísticos reproducibles__

Sobre las variables que mapean a `CAMELS`, aplicamos filtros estadisticos:

>- Eliminar variables con >50% NaN
>- Eliminar variables con varianza cuasi-cero
>- Eliminar variables con correlación >0.95 entre sí (VIF o matriz de correlación)


__Capa C — Importancia estadística con el target__
Calculas la correlación punto-biserial de cada variable candidata con el evento de quiebra del archivo `failures`. Las variables que selecionamos finalmente son las que tienen mayor poder discriminativo estadístico.

Ahora el procedimiento de preprocesado es el siguiente: Primero aplicamos la `capa B`: filtros estadisticos para la eliminación de variables nulas, eliminar columnas con varianza cuasi nula y eliminar columnas duplicada por nombre.

In [7]:
# recuperamos los paneles guardados
panels = {}
for name in ['FTS', 'CDI', 'RAT', 'MERG', 'STRU']:
    panels[name] = pd.read_parquet(PROCESSED_DIR / f"panel_{name}.parquet")
    print(f"{name}: {panels[name].shape}")

FTS: (206129, 3370)
CDI: (206129, 1098)
RAT: (206129, 254)
MERG: (1100520, 62)
STRU: (219576, 123)


A la hora de definir el umbral con el que vamos a trabajar para eliminar variables con alto porcentaje de `NaN`, no existe un valor definido claro, pero si tenemos en cuenta ciertos aspectos, en literatura de riesgo financiero y banking `50%` es el umbral más usado en papers de early warning systems bancarios (Demyanyk & Hasan, 2010; Betz et al., 2014) ya que en datos regulatorios bancarios, muchos campos son opcionales según el tipo de institución, lo cual implica que un `NaN` no siempre significa dato faltante, sino que esa variable no aplica a ese banco. En machine learning tabular general:

>80-90% es más permisivo — se usa cuando los NaN son aleatorios y no estructurales
>20-30% es muy estricto — se usa cuando tienes muchos datos y puedes permitirte perder variables

Luego tenemos que nuestro dataset de `RIS` tiene `NaN` estructurales, ya que recopila datos de instituciones y bancos  pequeños que no reportan variables de derivados financieros complejos, no porque le falte el dato sino porque no opera con ellos. Luego lo que hacemos es un breve estudio sobre cual es el mejor umbral para nuestros datos

In [8]:
print(f"{'Umbral':<10} {'FTS':>8} {'CDI':>8} {'RAT':>8} {'MERG':>8} {'STRU':>8}")
print("-" * 50)
for threshold in [0.3, 0.5, 0.6, 0.7, 0.8]:
    row = f"{threshold*100:.0f}%{'':<7}"
    for name in ['FTS', 'CDI', 'RAT', 'MERG', 'STRU']:
        # Columnas que SE ELIMINAN por tener más NaN que el umbral
        n_eliminadas = (panels[name].isnull().mean() > threshold).sum()
        row += f"{n_eliminadas:>8}"
    print(row)

Umbral          FTS      CDI      RAT     MERG     STRU
--------------------------------------------------
30%           1892     354      43       2       9
50%           1793     341      35       2       7
60%           1724     325      30       2       7
70%           1231     265      26       2       7
80%           1077     252      24       1       5


Como vemos en los paneles de `MERG` y `STRU` casi no cambian entre umbrales ya que son paneles pequeños, en comparación con los otros y bastante completos. En los demas a excepción de `FTS`, vemos que el cambio en los umbrales no es muy agresivo, en `CDI` pasar de un umbral de 0.3 a 0.8 solo elimina 103 columnas más. Donde se ve más el salto es en `FTS` ya que de 60% al 70% hay un salto de 483 columnas lo que indica que hay un bloque grande de columnas con un 60% minimo y un 70% máximo de `NaN` en sus datos, que probablemente sean variables opcionales que muy pocos bancos reportan.

In [9]:
def clean_panel(df: pd.DataFrame, name: str, 
                nan_threshold: float,
                variance_threshold: float = 0.0) -> pd.DataFrame:
    """
    Limpieza estadística en tres pasos:
    1. Elimina columnas con más del nan_threshold% de NaN
    2. Elimina columnas numéricas con varianza cero o cuasi-cero
    3. Elimina columnas duplicadas
    """
    original_cols = df.shape[1]
    
    # Paso 1 — Filtro NaN
    nan_rate = df.isnull().mean()
    cols_nan = nan_rate[nan_rate > nan_threshold].index.tolist()
    df = df.drop(columns=cols_nan)
    print(f"{name} | Paso 1 NaN >{nan_threshold*100:.0f}%: "
          f"-{len(cols_nan)} cols → {df.shape[1]} restantes")
    
    # Paso 2 — Filtro varianza cero (solo numéricas)
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    low_var = [c for c in num_cols if df[c].var() <= variance_threshold]
    df = df.drop(columns=low_var)
    print(f"{name} | Paso 2 varianza=0: "
          f"-{len(low_var)} cols → {df.shape[1]} restantes")
    
    # Paso 3 — Duplicadas por nombre (evita transposición)
    dupes = df.columns.duplicated().sum()
    df = df.loc[:, ~df.columns.duplicated()]
    print(f"{name} | Paso 3 nombres duplicados: "
          f"-{dupes} cols → {df.shape[1]} restantes")
    
    total = original_cols - df.shape[1]
    print(f"{name} | Total eliminadas: {total} de {original_cols} "
          f"({total/original_cols*100:.1f}%)\n")
    return df

In [10]:

THRESHOLD = 0.5  # umbral elegido tras el análisis

panels_clean = {}
for name, df in panels.items():
    print(f"{'='*45}")
    print(f"Limpiando {name}...")
    panels_clean[name] = clean_panel(df, name, nan_threshold=THRESHOLD)
    output = PROCESSED_DIR / f"panel_{name}_clean.parquet"
    panels_clean[name].to_parquet(output, index=False)
    print(f"Guardado: {output}")

print(f"\n{'Base':<8} {'Original':>10} {'Limpio':>10} {'Reducción':>12}")
print("-" * 42)
for name in ['FTS', 'CDI', 'RAT', 'MERG', 'STRU']:
    orig = panels[name].shape[1]
    clean = panels_clean[name].shape[1]
    pct = (orig - clean) / orig * 100
    print(f"{name:<8} {orig:>10} {clean:>10} {pct:>11.1f}%")

Limpiando FTS...
FTS | Paso 1 NaN >50%: -1793 cols → 1577 restantes
FTS | Paso 2 varianza=0: -20 cols → 1557 restantes
FTS | Paso 3 nombres duplicados: -0 cols → 1557 restantes
FTS | Total eliminadas: 1813 de 3370 (53.8%)

Guardado: D:\financial_risk_data\processed\panel_FTS_clean.parquet
Limpiando CDI...
CDI | Paso 1 NaN >50%: -341 cols → 757 restantes
CDI | Paso 2 varianza=0: -6 cols → 751 restantes
CDI | Paso 3 nombres duplicados: -0 cols → 751 restantes
CDI | Total eliminadas: 347 de 1098 (31.6%)

Guardado: D:\financial_risk_data\processed\panel_CDI_clean.parquet
Limpiando RAT...
RAT | Paso 1 NaN >50%: -35 cols → 219 restantes
RAT | Paso 2 varianza=0: -0 cols → 219 restantes
RAT | Paso 3 nombres duplicados: -0 cols → 219 restantes
RAT | Total eliminadas: 35 de 254 (13.8%)

Guardado: D:\financial_risk_data\processed\panel_RAT_clean.parquet
Limpiando MERG...
MERG | Paso 1 NaN >50%: -2 cols → 60 restantes
MERG | Paso 2 varianza=0: -0 cols → 60 restantes
MERG | Paso 3 nombres duplicado

`RAT` es el panel más limpio, solo el 13.8% eliminado y varianza cero en ninguna columna. Lo cual nos confirma que es la bse de datos más sólida para crear el panel temporal, ya que casi todas sus variables son informativas.

`FTS` pierde el 54.7% debido posiblemente a que es el panel más amplio y con muchas variables opcionales por tipo de institución. Los 36 de varianza cero son probablemente flags binarios que nunca cambiaron en el periodo.

`CDI` 31.9% presenta una reducción moderada, coherente con ser un panel de variables derivadas.

`MERG` y `STRU` casi intactos ya que son paneles que contienen datos estructurales.

El siguiente paso es hacer el estudio `CAMELS` sobre los paneles procesados de `FTS`, `RAT` y `CDI` que seran las bases de datos con las que construiremos nuestros datos tabulares para TabPFN. La idea es quedarse con las variables más relevantes siguiendo el estudio `CAMELS`, que es el framework oficial que usan los reguladores bancarios estadounidenses (FDIC, Fed, OCC) para evaluar la salud de un banco. Cada letra representa una dimensión de riesgo:

C — Capital Adequacy: ¿tiene el banco suficiente capital para absorber pérdidas?
A — Asset Quality: ¿qué porcentaje de sus préstamos son problemáticos?
M — Management: ¿opera el banco de forma eficiente?
E — Earnings: ¿es rentable?
L — Liquidity: ¿puede hacer frente a sus obligaciones a corto plazo?
S — Sensitivity to market risk: ¿cómo le afectan los cambios en tipos de interés y mercados?

Como ya hemos visto, tenemos un diccionario, faciliatdo por el `FDIC`, que no explica brevemente que significan cada variable de cada base de datos, luego para seleccionar las variables restantes a la limpieza mas relevantes lo que hacemos es un mapeo por keywords en las descripciones del diccionario oficial FDIC. Usamos las siguientes keywords ya que cada componente `CAMELS` tiene un significado financiero concreto, y las keywords son los términos técnicos que la `FDIC` usa en sus propias descripciones de variables.

C — Capital: las variables de capital se describen con términos como CAPITAL, EQUITY, TIER (Tier 1, Tier 2 son los niveles de capital regulatorio Basel III), RBC (Risk Based Capital), LEVERAGE y TANGIBLE (capital tangible, sin intangibles como goodwill).
A — Asset Quality: la calidad de activos se mide por morosidad y pérdidas. Los términos del diccionario son NONCURR (noncurrent = préstamos impagados), CHARGE-OFF (préstamos dados por perdidos), NONPERFORM (nonperforming assets), PAST DUE (vencidos), LOAN LOSS (pérdidas en préstamos), ALLOWANCE (provisiones) y PROVISION.
M — Management: la eficiencia operativa se describe con EFFICIENCY (el efficiency ratio es la métrica estándar — gastos operativos / ingresos), EXPENSE, OVERHEAD, EMPLOYEE (productividad por empleado).
E — Earnings: la rentabilidad usa INCOME, RETURN (ROA, ROE), PROFIT, EARNINGS, REVENUE, MARGIN (net interest margin) y directamente ROA, ROE.
L — Liquidity: la liquidez se mide por la estructura de financiación — DEPOSIT, LIQUID, CASH, BORROW, FUNDING y ratios como loans/deposits.
S — Sensitivity: el riesgo de mercado usa MARKET, TRADING, DERIVATIVE, RATE (sensibilidad a tipos de interés), FOREIGN (riesgo divisa), SECURITIES y HEDGE.

Las keywords no son una elección arbitraria sino que son los términos que aparecen literalmente en el diccionario oficial de la FDIC para describir variables que miden cada dimensión CAMELS.

In [11]:
import csv
# Cargar diccionarios
def load_dict(path):
    with open(path, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        return {r['RIS Name']: r['Description'] for r in reader}

# Rutas a los diccionarios (en tu máquina local)
# Necesitas tener los CSV del diccionario accesibles
# Si los tienes en el repo, ajusta la ruta
dict_rat = load_dict(project_root / "docs" / "RAT.csv")
dict_fts = load_dict(project_root / "docs" / "FTS.csv")
dict_cdi = load_dict(project_root / "docs" / "CDI.csv")

# Keywords por componente CAMELS
CAMELS_KEYWORDS = {
    "C": ["CAPITAL", "EQUITY", "TIER", "RBC", "LEVERAGE", "TANGIBLE", "SOLVENCY"],
    "A": ["NONCURR", "CHARGE-OFF", "NONPERFORM", "PAST DUE", 
          "LOAN LOSS", "ALLOWANCE", "PROVISION", "DELINQ"],
    "M": ["EFFICIENCY", "EXPENSE", "OVERHEAD", "EMPLOYEE", 
          "SALARY", "NONINTEREST EXP"],
    "E": ["INCOME", "RETURN", "PROFIT", "EARNINGS", 
          "REVENUE", "MARGIN", "ROA", "ROE"],
    "L": ["DEPOSIT", "LIQUID", "CASH", "BORROW", 
          "FUNDING", "ASSET/DEP", "LOANS/DEP"],
    "S": ["MARKET", "TRADING", "DERIVATIVE", "RATE", 
          "FOREIGN", "SECURITIES", "HEDGE"],
}

def map_to_camels(description: str) -> str:
    """Asigna un componente CAMELS a una variable por su descripción."""
    desc_upper = description.upper()
    matches = []
    for component, keywords in CAMELS_KEYWORDS.items():
        if any(k in desc_upper for k in keywords):
            matches.append(component)
    if len(matches) == 1:
        return matches[0]
    elif len(matches) > 1:
        return matches[0]  # primer match gana — orden CAMELS
    return "UNASSIGNED"

# Cargar columnas supervivientes de cada panel limpio
rat_clean = pd.read_parquet(PROCESSED_DIR / "panel_RAT_clean.parquet")
fts_clean = pd.read_parquet(PROCESSED_DIR / "panel_FTS_clean.parquet")
cdi_clean = pd.read_parquet(PROCESSED_DIR / "panel_CDI_clean.parquet")

# Construir mapping completo
records = []
for source, clean_df, dictionary in [
    ("RAT", rat_clean, dict_rat),
    ("FTS", fts_clean, dict_fts),
    ("CDI", cdi_clean, dict_cdi),
]:
    nan_rates = clean_df.isnull().mean()
    for col in clean_df.columns:
        if col in ["CERT", "period", "CALLYM", "CALLYMD", "REPDTE"]:
            continue  # skip identificadores
        desc = dictionary.get(col, "")
        component = map_to_camels(desc)
        records.append({
            "variable":  col,
            "source":    source,
            "description": desc,
            "camels":    component,
            "nan_rate":  round(nan_rates[col], 4),
        })

df_mapping = pd.DataFrame(records)

# Guardar
output_path = project_root / "docs" / "camels_mapping.csv"
df_mapping.to_csv(output_path, index=False)
print(f"Mapping guardado: {output_path}")
print(f"\nTotal variables mapeadas: {len(df_mapping)}")
print(f"\nDistribución por componente:")
print(df_mapping['camels'].value_counts())
print(f"\nDistribución por fuente:")
print(df_mapping.groupby(['source', 'camels']).size().unstack(fill_value=0))

Mapping guardado: c:\dev\tfm-financial_risk\docs\camels_mapping.csv

Total variables mapeadas: 2512

Distribución por componente:
camels
UNASSIGNED    1877
E              161
S              128
C              115
L              102
A               73
M               56
Name: count, dtype: int64

Distribución por fuente:
camels   A   C   E   L   M    S  UNASSIGNED
source                                     
CDI     16  25  83  39  26   23         534
FTS     41  61  49  54  16  101        1230
RAT     16  29  29   9  14    4         113


Ya temos las variables segun el framework de `CAMELS`, ahora es ver cuales son explcitamente

In [12]:
df_mapping = pd.read_csv(project_root / "docs" / "camels_mapping.csv")

# Variables CAMELS por fuente (exluimos UNASSIGNED)
camels_assigned = df_mapping[df_mapping['camels'] != 'UNASSIGNED']

for source in ['RAT', 'FTS', 'CDI']:
    print(f"\n{'='*50}")
    print(f"{source} — variables CAMELS asignadas:")
    subset = camels_assigned[camels_assigned['source'] == source]
    for component in ['C', 'A', 'M', 'E', 'L', 'S']:
        vars_comp = subset[subset['camels'] == component][['variable', 'description', 'nan_rate']]
        print(f"\n  {component} ({len(vars_comp)}):")
        print(vars_comp.to_string(index=False))


print("Variables CAMELS asignadas por panel:")
print(camels_assigned.groupby('source').size())
print(f"\nTotal: {len(camels_assigned)}")


RAT — variables CAMELS asignadas:

  C (29):
variable                     description  nan_rate
EQTANQTA   TANGIBLE EQUITY CAPITAL RATIO    0.0032
 EQTOTY1         TOTAL EQUITY CAPITAL-Y1    0.0050
     EQV      BANK EQUITY CAPITAL/ASSETS    0.0032
    EQY1    TOTAL BANK EQUITY CAPITAL-Y1    0.0050
   EQY1S  TOTAL BANK EQUITY CAPITAL-S-Y1    0.0050
LIABEQY1  TOTAL LIABILITIES & CAPITAL-Y1    0.0017
 LNAGT1R                 AG LOANS/TIER 1    0.0032
 LNCDT1R    CONSTR & LAND DEV LNS/TIER 1    0.0032
 LNCIT1R                C&I LOANS/TIER 1    0.0032
LNCONT1R           CONSUMER LOANS/TIER 1    0.0032
 LNHRSKR          HIGH RISK LOANS/TIER 1    0.0032
LNRERT1R                 RE LOANS/TIER 1    0.0032
NCRELOCR     N/C HOME EQUITY/HOME EQUITY    0.2663
NTRELOCR HOME EQUITY CHG-OFF/HOME EQ LNS    0.2586
P3RELOCR   30-89 P/D HOME EQUITY/HOME EQ    0.2663
 RBC1AAJ              LEVERAGE RATIO-PCA    0.0032
RBC1AAJP              LEVERAGE RATIO-RBP    0.0032
 RBC1AAW         LEVERAGE RATIO-REPO

Las variables CAMELS asignadas por panel son:
```p
CDI    212
FTS    322
RAT    101
```

Lo que hace un total de 635 variables, lo cual supone una reducción bastante aceptable. Ahora el siguiente y ultimo paso es realizar un estudio de la correlacion entre todas estas variables para excluir posibles correlaciones entre distintas variables de los tres paneles. El proceso sera unir las 635 variables `CAMELS` de los tres paneles en un solo `DataFrame` mediante la union de `CERT + period`, luego calcular matriz de correlación completa y eliminar variables con correlación >0.95.

In [13]:
# Variables CAMELS por panel
camels_fts = df_mapping[(df_mapping['source']=='FTS') & 
                         (df_mapping['camels']!='UNASSIGNED')]['variable'].tolist()
camels_cdi = df_mapping[(df_mapping['source']=='CDI') & 
                         (df_mapping['camels']!='UNASSIGNED')]['variable'].tolist()
camels_rat = df_mapping[(df_mapping['source']=='RAT') & 
                         (df_mapping['camels']!='UNASSIGNED')]['variable'].tolist()

print(f"Variables CAMELS — FTS: {len(camels_fts)} | CDI: {len(camels_cdi)} | RAT: {len(camels_rat)}")

# Cargar solo las columnas necesarias de cada panel
cols_fts = ['CERT', 'period'] + camels_fts
cols_cdi = ['CERT', 'period'] + camels_cdi
cols_rat = ['CERT', 'period'] + camels_rat

fts = pd.read_parquet(PROCESSED_DIR / "panel_FTS_clean.parquet", columns=cols_fts)
cdi = pd.read_parquet(PROCESSED_DIR / "panel_CDI_clean.parquet", columns=cols_cdi)
rat = pd.read_parquet(PROCESSED_DIR / "panel_RAT_clean.parquet", columns=cols_rat)

print(f"FTS: {fts.shape} | CDI: {cdi.shape} | RAT: {rat.shape}")

Variables CAMELS — FTS: 322 | CDI: 212 | RAT: 101
FTS: (206129, 324) | CDI: (206129, 214) | RAT: (206129, 103)


In [14]:

panel_combined = fts.merge(rat, on=['CERT','period'], how='inner', suffixes=('_fts','_rat'))
panel_combined = panel_combined.merge(cdi, on=['CERT','period'], how='inner', suffixes=('','_cdi'))

print(f"Panel combinado: {panel.shape}")
print(f"Periodos: {panel['period'].nunique()} | CERT únicos: {panel['CERT'].nunique()}")

print("Tipos de dato en panel combinado:")
print(panel_combined.dtypes.value_counts())

# Ver ejemplos de columnas string
str_cols = [c for c in panel_combined.columns 
            if str(panel_combined[c].dtype) in ['object', 'string']]
print(f"\nColumnas string: {len(str_cols)}")

Panel combinado: (219576, 123)
Periodos: 40 | CERT únicos: 6638
Tipos de dato en panel combinado:
str        536
float64    100
int64        1
Name: count, dtype: int64

Columnas string: 0


A la hora de explorar el tipo de datos que tiene nuestro panel combinado, tenemos 536 variables como `str` pero `select_dtypes` no las encuentra como `object`. Esto se debe porque pandas 3.0.3 que es el que estamos usando introduce un nuevo tipo `str` nativo distinto de `object`.

In [15]:
# En pandas 3.0.3 el tipo str es distinto de object
str_cols = [c for c in panel_combined.columns 
            if str(panel_combined[c].dtype) == 'str']
print(f"Columnas tipo str: {len(str_cols)}")
print(f"Ejemplos: {str_cols[:5]}")

# Ver valores 
if str_cols:
    print(f"\nValores de {str_cols[0]}:")
    print(panel_combined[str_cols[0]].head(10))
    print(f"\nSon numéricos? Intentando conversión...")
    converted = pd.to_numeric(panel_combined[str_cols[0]], errors='coerce')
    print(f"NaN tras conversión: {converted.isna().sum()} de {len(converted)}")
    

Columnas tipo str: 536
Ejemplos: ['CERT', 'period', 'AVASSETW', 'AVOTHBOR', 'AVSAVDP']

Valores de CERT:
0    14
1    35
2    39
3    41
4    46
5    47
6    49
7    50
8    51
9    52
Name: CERT, dtype: str

Son numéricos? Intentando conversión...
NaN tras conversión: 0 de 206129


Esto ultimo nos indican que estan todas bien, `CERT` y `period` son `string` por diseño, y el resto son numéricas almacenadas como `str` de pandas 3.0.3, que se pueden convertir a numérico sin ningún `NaN` adicional. Luego tenemos que convertir todas las columnas `str` a numéricas antes de calcular la correlación

In [16]:
id_cols = ['CERT', 'period']

# Convertir columnas str numéricas a float64
for col in str_cols:
    if col in id_cols:
        continue
    panel_combined[col] = pd.to_numeric(panel_combined[col], errors='coerce')

print("Tipos tras conversión:")
print(panel_combined.dtypes.value_counts())

# Ahora sí — correlación sobre todas las numéricas
num_cols = panel_combined.drop(columns=id_cols).select_dtypes(include=['float64','int64']).columns.tolist()
print(f"\nColumnas numéricas para correlación: {len(num_cols)}")

Tipos tras conversión:
float64    634
str          2
int64        1
Name: count, dtype: int64

Columnas numéricas para correlación: 635


In [17]:
# Matriz de correlación absoluta y eliminación de redundantes

# Muestra para correlación (635 columnas x 40k filas)
sample = panel_combined[num_cols].sample(n=40000, random_state=42)
corr_matrix = sample.corr().abs()

# Triangulo superior para evitar duplicados
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]

print(f"Variables a eliminar por correlación >0.95: {len(to_drop)}")

# Origen de las eliminadas
drop_sources = df_mapping[df_mapping['variable'].isin(to_drop)]['source'].value_counts()
print(f"\nOrigen de las variables eliminadas:\n{drop_sources}")

# Variables que sobreviven
final_cols = [c for c in num_cols if c not in to_drop]
print(f"\nVariables finales: {len(final_cols)}")

print(f"\nVariables finales por componente CAMELS:")
for comp in ['C','A','M','E','L','S']:
    vars_comp = df_mapping[
        (df_mapping['variable'].isin(final_cols)) & 
        (df_mapping['camels']==comp)
    ]
    print(f"  {comp}: {len(vars_comp)}")

Variables a eliminar por correlación >0.95: 226

Origen de las variables eliminadas:
source
CDI    112
FTS     86
RAT     28
Name: count, dtype: int64

Variables finales: 409

Variables finales por componente CAMELS:
  C: 57
  A: 62
  M: 40
  E: 95
  L: 70
  S: 85


- `CDI` es el panel más redundante, 112 de sus variables eliminadas por correlación >0.95 con `FTS` o `RAT`. Esto valida empíricamente la decisión de excluirlo ya que el 53% de las variables `CAMELS` de `CDI` presentan correlación superior a 0.95 con variables equivalentes en `FTS` o `RAT`, confirmando su naturaleza derivada.

- `FTS` pierde 86 debido a desgloses internos muy correlacionados entre sí.
- `RAT` pierde 28 debido versiones anuales y trimestrales del mismo ratio.

409 variables finales es perfectamente manejable para `TabPFN` dentro del límite de 2.000 features con margen amplio.

In [18]:
panel_temporal = panel_combined[id_cols + final_cols].copy()

print(f"Panel final: {panel_temporal.shape}")
panel_temporal.to_parquet(PROCESSED_DIR / "panel_temporal.parquet", index=False)
print(f"Guardado: panel_temporal.parquet")

# Guardar también la lista de variables seleccionadas
df_selected = df_mapping[df_mapping['variable'].isin(final_cols)].copy()
df_selected.to_csv(project_root / "docs" / "camels_selected.csv", index=False)
print(f"Guardado: camels_selected.csv")

Panel final: (206129, 411)
Guardado: panel_temporal.parquet
Guardado: camels_selected.csv


La dimension del panel final es (206129, 411). Aunque tomamos 409 variables de las bases de datos individuales, se debe a que tambien incluimos las dos variables `CERT` y `period`.

In [19]:
df_selected = pd.read_csv(project_root / "docs" / "camels_selected.csv")

for source in ['FTS', 'CDI', 'RAT']:
    subset = df_selected[df_selected['source'] == source]
    print(f"\n{source} ({len(subset)} variables):")
    for comp in ['C','A','M','E','L','S']:
        vars_comp = subset[subset['camels'] == comp]['variable'].tolist()
        if vars_comp:
            print(f"  {comp}: {vars_comp}")


FTS (236 variables):
  C: ['APCDLNLS', 'APCDOFA', 'APCDSCHM', 'AVASSETW', 'AVTANEQ', 'CCED1L', 'CCED1T5', 'CCEDOV5', 'CT1BADJ', 'CT1MIN', 'ED1LES', 'ED1T5', 'EDOVR5', 'EQCSTKRX', 'EQOTHCC', 'INTAN', 'INTANTO', 'LIABEQ', 'RB2LNRES', 'RBASTDL', 'RBCT1ADD', 'RBOTHDL', 'RWATOTW', 'SCDBEQ', 'SCEQ', 'SCFDEQ', 'T1ADCPIN', 'T2AFSGN', 'T2CAPINS', 'T2DED', 'T2MIN', 'T2NQCAP', 'UCLOC']
  A: ['DRAGSM', 'DRAUTO', 'DRCI', 'DRCOMRE', 'DRCON', 'DRCONOTH', 'DRCRCD', 'DRLNLS', 'DRLS', 'DROTHER', 'DRRE', 'DRREAG', 'DRRECNFM', 'DRRECNOT', 'DRRECONS', 'DRRELOC', 'DRREMULT', 'DRRENRES', 'DRREOT', 'DRREOTH', 'DRRERES', 'ELNATR', 'LNLSRES', 'NCGTYGNM', 'NTAGSM', 'NTCI', 'NTCOMRE', 'NTCON', 'NTCONOTH', 'NTLNLS', 'NTLS', 'NTOTHER', 'NTRE', 'NTREAG', 'NTREOT']
  M: ['ECD100', 'EDEP', 'EFREPP', 'EOTHINT', 'EOTHNINT', 'EOTHTIME', 'EPREMAGG', 'ESAL', 'ETRANDEP', 'NETNIX', 'NONIX', 'NUMEMP', 'OLINT', 'OLINTOTH']
  E: ['EQCCOMPI', 'EQCOMINC', 'EQUP', 'EQUPGR', 'IBEFTAX', 'IFIDUC', 'IFREPO', 'ILN', 'ILNAG', 'ILNCI', 

Ahora tenemos que crear el panel de nodos del grafo, que lo haremos uniendo las variables que tenemos seleccionadas en el `utils/config.py`. Debido a que `STRU` contiene información estructural y no financiera de la institución, como localización, agencia reguladora, charter y características básicas derivadas de otras fuentes institucionales y `MERG` recoge datos no financieros de fusiones y cambios estructurales, derivados de INST y otras fuentes externas; sirve para capturar discontinuidades y eventos institucionales, podemos decidir cuales tomar basandonos simplemente en la información disponible del diccionario de RSI y de las columnas disponibles tras la limpieza previa de las dos bases de datos.

In [20]:

from src.utils.config import STRU_SELECTED, MERG_SELECTED
# Cargar solo columnas seleccionadas
stru = pd.read_parquet(PROCESSED_DIR / "panel_STRU_clean.parquet")
merg = pd.read_parquet(PROCESSED_DIR / "panel_MERG_clean.parquet")

# Filtrar solo columnas que existen en el panel
stru_cols = [c for c in STRU_SELECTED if c in stru.columns]
merg_cols = [c for c in MERG_SELECTED if c in merg.columns]

stru_miss = [c for c in STRU_SELECTED if c not in stru.columns]
merg_miss = [c for c in MERG_SELECTED if c not in merg.columns]

print(f"STRU: {len(stru_cols)}/{len(STRU_SELECTED)} cols | Missing: {stru_miss}")
print(f"MERG: {len(merg_cols)}/{len(MERG_SELECTED)} cols | Missing: {merg_miss}")

stru = stru[stru_cols]
merg = merg[merg_cols]

print(stru.shape, merg.shape)

STRU: 18/18 cols | Missing: []
MERG: 8/8 cols | Missing: []
(219576, 18) (1100520, 8)


Problema 1 — Granularidad temporal y acumulación histórica. MERG no utiliza el campo period como el resto de paneles sino YEARQTR, un entero de 5 dígitos con formato YYYYQ (por ejemplo 20173 = tercer trimestre de 2017). A diferencia de FTS, CDI o RAT que solo contienen datos del trimestre reportado, cada corte trimestral de MERG es acumulativo — incluye el historial completo de eventos de fusión y cierre hasta esa fecha. Esto genera duplicidades al concatenar los 40 trimestres: el mismo evento aparece en múltiples cortes. La solución es filtrar por YEARQTR para quedarse solo con los eventos ocurridos en nuestro periodo de análisis (2016Q1-2025Q4) y derivar period mediante un mapeo del último dígito al formato estándar YYYYQX.

El problema que tenemos con `MERG` que a diferencia de las otras bases de datos es que `CERT` en `MERG` es siempre 00000, luego no es el identificador del banco superviviente. `CERT` es siempre 00000 en el raw que es así en la fuente FDIC. La estructura real es:

- C_CERT = banco adquirido/cerrado (35367, 18923...)
- L_CERT = último certificado asociado (35367, 18923...) — mismo valor que C_CERT en estos casos
- CERT = siempre 00000 — campo vacío

No tenemos el identificador del banco adquirente en este archivo. MERG solo registra el lado del banco que desaparece, no el que sobrevive. MERG lo usamos como señal de evento sobre el banco que cierra — no como arista entre dos bancos activos. Su valor en el grafo es: Saber qué bancos cerraron en cada trimestre, Cruzarlo con failures para enriquecer la señal de riesgo y Usar C_CERT + period como clave de agregación


Problema 2 — Identificador del banco superviviente ausente. El campo CERT en MERG es siempre 00000 — la FDIC no registra el banco adquirente en este archivo. Según el diccionario oficial, MERG registra exclusivamente el lado del banco que desaparece: C_CERT identifica al banco adquirido (serie 200) y no existe una columna equivalente para el adquirente (serie 800). Por tanto MERG no puede usarse para construir aristas directas entre banco adquirente y adquirido — solo aporta información sobre el banco que cierra.
Problema 3 — Múltiples eventos por banco por trimestre. MERG tiene 1.100.520 filas para 40 trimestres, frente a las ~219.000 de STRU. Un banco puede tener varios registros en el mismo trimestre. Para poder hacer el join con STRU se agrega MERG por C_CERT + period, resumiendo los eventos en cinco variables: número de eventos de cierre, flag de banco fallido, flag de asistencia FDIC, activos totales en el momento del cierre y depósitos totales en el momento del cierre.
Construcción del panel final. STRU actúa como base con left join sobre MERG agregado, usando CERT + period como clave. Los bancos sin eventos de cierre en ese trimestre reciben valor 0 en las variables de MERG. El panel resultante tiene 219.576 filas, 40 periodos y 6.638 bancos únicos con 23 variables por nodo.
Los NaN residuales en variables como CBSA (57.466) y RSSDHCR (49.905) son estructuralmente informativos — indican bancos rurales sin área metropolitana asignada y bancos independientes sin holding company respectivamente — y se tratan como categoría propia en la construcción del grafo.

`STRU` es la base del panel de nodos — contiene la identidad, geografía y estructura de cada banco activo en cada trimestre. `MERG` lo cruzamos como señal auxiliar: si un banco aparece en MERG ese trimestre, añadimos un flag de evento de cierre.

In [21]:
# 1 — Cargar STRU limpio y con columnas filtradas
stru = pd.read_parquet(PROCESSED_DIR / "panel_STRU_clean.parquet")
stru_cols = [c for c in STRU_SELECTED if c in stru.columns]
stru = stru[stru_cols].copy()
print(f"STRU: {stru.shape}")

# 2 — Cargar MERG, filtrar por columnas, deduplicar y preparar
merg = pd.read_parquet(PROCESSED_DIR / "panel_MERG_clean.parquet")
merg_cols = [c for c in MERG_SELECTED if c in merg.columns]

# Deduplicar primero — cada evento único por C_CERT + EFFDATE
merg = merg[merg_cols].copy()  # filtrar antes de deduplicar
merg = merg.drop_duplicates(subset=['C_CERT', 'EFFDATE']).copy()
print(f"MERG tras deduplicar: {merg.shape}")

# Limpiar tipos numéricos
merg['L_ASSET'] = pd.to_numeric(
    merg['L_ASSET'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)
merg['L_DEP'] = pd.to_numeric(
    merg['L_DEP'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)

# Filtrar nuestro periodo y derivar period
merg = merg[merg['YEARQTR'] >= 20161].copy()
quarter_map = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}
merg['period'] = merg['YEARQTR'].astype(str).apply(
    lambda x: x[:4] + quarter_map[int(x[4])]
)
print(f"MERG en nuestro periodo: {merg.shape}")

# 3 — Agregar por C_CERT + period
merg_agg = merg.groupby(['C_CERT', 'period']).agg(
    closure_event=('EFFDATE', 'count'),
    closure_type=('CODE2XX', 'first'),   # tipo de cierre predominante
    fdic_assisted=('ASSIST', 'max'),
    last_assets=('L_ASSET', 'max'),
    last_deposits=('L_DEP', 'max'),
).reset_index().rename(columns={'C_CERT': 'CERT'})

merg_agg['CERT'] = merg_agg['CERT'].astype(str)
print(f"MERG agregado: {merg_agg.shape}")

# 4 — Join STRU + MERG
panel_nodos = stru.merge(merg_agg, on=['CERT', 'period'], how='left')

# Rellenar NaN de MERG con 0
panel_nodos[['closure_event', 'fdic_assisted',
             'last_assets', 'last_deposits']] = \
    panel_nodos[['closure_event', 'fdic_assisted',
                 'last_assets', 'last_deposits']].fillna(0)
panel_nodos['closure_type'] = panel_nodos['closure_type'].fillna(0)

print(f"\nPanel nodos: {panel_nodos.shape}")
print(f"Periodos: {panel_nodos['period'].nunique()}")
print(f"CERT únicos: {panel_nodos['CERT'].nunique()}")
print(f"\nNaN restantes:")
print(panel_nodos.isnull().sum()[panel_nodos.isnull().sum() > 0])

# 5 — Guardar
panel_nodos.to_parquet(PROCESSED_DIR / "panel_nodos.parquet", index=False)
print(f"\nGuardado: panel_nodos.parquet")

STRU: (219576, 18)
MERG tras deduplicar: (27825, 8)
MERG en nuestro periodo: (2133, 9)
MERG agregado: (2121, 7)

Panel nodos: (219576, 23)
Periodos: 40
CERT únicos: 6638

NaN restantes:
STALP         1160
SIMS_LAT      1324
SIMS_LONG     1324
CBSA         57466
METRO        57466
RSSDHCR      49905
OFFDOM        1823
OFFTOT        1823
OFFSTATE      1110
dtype: int64

Guardado: panel_nodos.parquet


In [22]:
# Ver cuántos eventos únicos hay por YEARQTR en nuestro periodo
merg_raw = pd.read_parquet(PROCESSED_DIR / "panel_MERG_clean.parquet")
merg_raw = merg_raw[merg_raw['YEARQTR'] >= 20161].copy()

# Cada evento único se identifica por C_CERT + EFFDATE
eventos_unicos = merg_raw.drop_duplicates(subset=['C_CERT', 'EFFDATE'])
print(f"Registros totales en nuestro periodo: {len(merg_raw)}")
print(f"Eventos únicos (C_CERT + EFFDATE): {len(eventos_unicos)}")
print(f"\nDistribución por YEARQTR:")
print(eventos_unicos['YEARQTR'].value_counts().sort_index().head(10))

Registros totales en nuestro periodo: 49920
Eventos únicos (C_CERT + EFFDATE): 2133

Distribución por YEARQTR:
YEARQTR
20161    71
20162    65
20163    83
20164    73
20171    63
20172    72
20173    57
20174    75
20181    72
20182    68
Name: count, dtype: int64


MERG es un archivo acumulativo — cada corte trimestral contiene el historial completo de eventos de fusión y cierre hasta esa fecha, generando 1.100.520 registros totales con alta duplicidad. Tras deduplicar por C_CERT + EFFDATE quedan 27.825 eventos únicos, de los cuales 2.133 corresponden a nuestro periodo de análisis (2016Q1-2025Q4). Estos se agregan por banco cerrado y trimestre produciendo 2.121 registros que se unen al panel STRU mediante left join

Finalmente, de esta forma de las 4902 variables (columnas) conjuntas que tenia nuestro dataset RIS, hemos creado dos paneles separados con las bases de datos individuales de cada tipo

- Definimos un panel temporal (`panel_temporal.parquet`) para crear datos tabulares temporales con los que alimetnar nuestro modelo `TabPFN` con una dimension de (206.129, 411), lo cual entra en los margenes de dimensionalidad de columnas definido por la aquitectura de TabPFN V2.5
- Definimos un panel de nodos (`panel_nodos.parquet`) para crear los nodos y aristas de nuestro grafo temporal `T_GCN` con dimension (219.576, 23), el cual lo ingestaremos para una mayor riqueza en los nodos, con los embeddings obtenidos de TabPFN

De esta forma tenemos los datos base para nuestro modelo de hibridación, donde hemos pasado de 4902 caracteristicas conjuntas a 442 caracteristicas separadas en dos paneles diferentes, es decir nos quedamos con un 9.017% de las caracteristicas iniciales, logrando asi una reduccion significativa.

Tenemos 409 variables `CAMELS` seleccionadas pero no todas discriminan igual entre bancos sanos y bancos que quiebran. La correlación `punto-biserial` nos da un ranking estadístico objetivo de cuáles son más informativas para predecir riesgo. La correlación `punto-biserial` es una medida estadística que calcula la correlación entre una variable continua (por ejemplo `ROA`, `NTLNLSR`) y una variable binaria (quebró = 1, no quebró = 0). Es exactamente la misma que la correlación de `Pearson` pero adaptada cuando una de las variables es dicotómica. El resultado va de -1 a +1 — cuanto más alto en valor absoluto, más discriminativa es la variable.

Esta correlación cumple con dos funciones: 
1. Valida empíricamente la selección `CAMELS`, si las variables con mayor correlación coinciden con los componentes `CAMELS` más relevantes teóricamente (A y C principalmente), la selección queda doblemente justificada.
2. Permite una reducción adicional opcional, en caso de que quisieramos acelerar o hacer que `TabPFN` trabaje con aun menos variables podríamos quedarnos con el _top-150_ o _top-200_ por correlación estadística.

Esta correlación `punto-biseria`l la hacemos con una base de datos llamada `failures`, tambien sacada de `RIS`, la cual se refiere al registro de instituciones financieras que han fracasado, sido intervenidas o cerradas y que luego se usan como referencia para análisis de riesgo y predicción de quiebra. Primero hagamos un pequeño estudio de la estructura de esta base de datos.

In [23]:
from src.data.loader import load_failures

failures = load_failures()
print(f"Shape: {failures.shape}")
print(f"\nColumnas: {failures.columns.tolist()}")
print(f"\nTipos:")
print(failures.dtypes)
print(f"\nPrimeras filas clave:")
print(failures[['CERT', 'FAILDATE', 'FAILYR', 'NAME', 'RESTYPE']].head(10))
print(f"\nRango temporal:")
print(f"  Primer fallo: {failures['FAILDATE'].min()}")
print(f"  Último fallo: {failures['FAILDATE'].max()}")
print(f"\nCERTs únicos: {failures['CERT'].nunique()}")
print(f"\nFallos por año (nuestro periodo 2016-2025):")
print(failures[failures['FAILYR'] >= 2016]['FAILYR'].value_counts().sort_index())
print(f"\nTipos de resolución:")
print(failures['RESTYPE'].value_counts())

Shape: (561, 33)

Columnas: ['RESDATE', 'QBFDEP', 'BIDCITY', 'BIDSTATE', 'CLOSCD', 'PSTALP', 'BSTATUS', 'COMMENTS', 'CITYST', 'RESTYPE', 'COSTMOSTRECENTASOF', 'URL', 'PTRDATE', 'COST', 'QBFASSET', 'UNINSDEP', 'TERMI', 'FAILYR', 'CHCLASS', 'FUND', 'BANKNO', 'BIDNAME', 'FIN', 'FAILDATE', 'SAVR', 'RESTYPE1', 'NAME', 'CHCLASS1', 'FSL_PROG', 'CITY', 'CERT', 'BRDATE', 'ID']

Tipos:
RESDATE                   str
QBFDEP                  int64
BIDCITY                   str
BIDSTATE                  str
CLOSCD                    str
PSTALP                    str
BSTATUS                   str
COMMENTS                  str
CITYST                    str
RESTYPE                   str
COSTMOSTRECENTASOF        str
URL                       str
PTRDATE                 int64
COST                  float64
QBFASSET                int64
UNINSDEP              float64
TERMI                     str
FAILYR                  int64
CHCLASS                   str
FUND                    int64
BANKNO               

Los datos arrojan la siguiente información: 561 bancos fallidos en total (ya que la base de datos toma valores desde el 2007), de los cuales 30 son en nuestro periodo de 2016-2025, lo cual indica un número muy pequeño sobre 6.282 bancos únicos. Esto confirma un desbalanceo de clases (~0.5% positivos).

`FAILDATE` es `string` en formato `M/D/YYYY`, luego hay que parsearlo antes de usarlo para tener la misma consistencia en las fechas con los demas paneles.

La base de datos está ordenanda alfabéticamente como string, no cronológicamente, por eso el rango temporal aparece tal que: (1/11/2013 a 9/9/2011).

`RESTYPE` tiene dos valores: `FAILURE` (548) y `ASSISTANCE` (13), donde este último indica los bancos que recibieron ayuda sin cerrar formalmente. Para el target binario usaremos ambos como evento de riesgo.

`CERT` es `int64`, luego hay que convertirlo a `string` para hacer el `join` con el `panel_temporal` ya que hemos estado tratando el `CERT` (identificador de banco) como `string`.

Antes de construir el flag binario necesitamos es el horizonte de predicción, es decir queremos predecir si un banco quiebra en el trimestre actual (t) o en los próximos N trimestres. Lo estándar en literatura de _early warning_ es 4-8 trimestres (1-2 años). El razonamiento es que un horizonte menor (1-2 trimestres) da poco margen de actuación regulatoria, y uno mayor (>8 trimestres) introduce demasiado ruido ya que las condiciones financieras cambian mucho en 2 años. Por tanto siguiendo la convención establecida en literatura de _early warning systems_ bancarios (Cole & Gunther, 1995; Betz et al., 2014), se define un horizonte de predicción de 4 trimestres. Una observación (`CERT`, `period`) se etiqueta como positiva (failed=1) si el banco quebró en alguno de los 4 trimestres siguientes al periodo observado.

In [24]:
# EDA failures ya hecho — construir flag binario
failures = load_failures()

# 1 — Parsear FAILDATE y convertir CERT a string
failures['FAILDATE'] = pd.to_datetime(failures['FAILDATE'], format='%m/%d/%Y')
failures['CERT'] = failures['CERT'].astype(str)

# Quedarnos solo con lo necesario
failures_clean = failures[['CERT', 'FAILDATE', 'FAILYR', 'RESTYPE']].copy()
print(f"Failures limpios: {failures_clean.shape}")
print(failures_clean.head(3))


# 2 — Construir mapa de periodos
# Necesitamos saber el orden de los periodos para calcular horizonte
panel = pd.read_parquet(PROCESSED_DIR / "panel_temporal.parquet",
                        columns=['CERT', 'period'])

# Lista ordenada de periodos únicos
periodos = sorted(panel['period'].unique())
periodo_idx = {p: i for i, p in enumerate(periodos)}
print(f"Periodos: {periodos}")


# 3 — Para cada banco en failures asignar el periodo de quiebra
def date_to_period(date):
    year = date.year
    month = date.month
    if month <= 3:
        return f"{year}Q1"
    elif month <= 6:
        return f"{year}Q2"
    elif month <= 9:
        return f"{year}Q3"
    else:
        return f"{year}Q4"

failures_clean['fail_period'] = failures_clean['FAILDATE'].apply(date_to_period)
print(f"\nFallos en nuestro periodo:")
print(failures_clean[failures_clean['fail_period'].isin(periodos)][
    ['CERT', 'fail_period', 'FAILYR', 'RESTYPE']
])


# 4 — Construir flag binario con horizonte 4 trimestres
HORIZON = 4

# Para cada (CERT, period) en el panel — failed=1 si el banco
# quiebra en los próximos HORIZON trimestres
failed_certs = set(failures_clean[
    failures_clean['fail_period'].isin(periodos)
]['CERT'].tolist())

def get_fail_period(cert):
    row = failures_clean[failures_clean['CERT'] == cert]
    if len(row) == 0:
        return None
    return row['fail_period'].values[0]

# Construir diccionario cert -> indice del periodo de quiebra
fail_period_map = {}
for _, row in failures_clean[failures_clean['fail_period'].isin(periodos)].iterrows():
    fail_period_map[row['CERT']] = periodo_idx.get(row['fail_period'], None)

# Asignar flag
def assign_failed_flag(row):
    cert = row['CERT']
    if cert not in fail_period_map:
        return 0
    fail_idx = fail_period_map[cert]
    if fail_idx is None:
        return 0
    current_idx = periodo_idx.get(row['period'], None)
    if current_idx is None:
        return 0
    # failed=1 si la quiebra ocurre en los próximos HORIZON trimestres
    if 0 < (fail_idx - current_idx) <= HORIZON:
        return 1
    return 0

print("Calculando flags puede tardar unos segundos...")
panel['failed'] = panel.apply(assign_failed_flag, axis=1)

print(f"\nDistribución del target:")
print(panel['failed'].value_counts())
print(f"\nTasa de positivos: {panel['failed'].mean()*100:.2f}%")

# Guardar panel con flag
panel.to_parquet(PROCESSED_DIR / "panel_temporal_labeled.parquet", index=False)
print("Guardado: panel_temporal_labeled.parquet")

Failures limpios: (561, 4)
    CERT   FAILDATE  FAILYR  RESTYPE
0  35353 2007-02-02    2007  FAILURE
1  32575 2007-09-28    2007  FAILURE
2  16848 2007-10-04    2007  FAILURE
Periodos: ['2016Q1', '2016Q2', '2016Q3', '2016Q4', '2017Q1', '2017Q2', '2017Q3', '2017Q4', '2018Q1', '2018Q2', '2018Q3', '2018Q4', '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4']

Fallos en nuestro periodo:
      CERT fail_period  FAILYR  RESTYPE
531  20364      2016Q1    2016  FAILURE
532   9956      2016Q2    2016  FAILURE
533  35312      2016Q2    2016  FAILURE
534  11297      2016Q3    2016  FAILURE
535     91      2016Q3    2016  FAILURE
536  34951      2017Q1    2017  FAILURE
537  19328      2017Q1    2017  FAILURE
538  35495      2017Q1    2017  FAILURE
539  58302      2017Q2    

30 bancos fallidos en nuestro periodo, pero con horizonte de 4 trimestres generamos 103 observaciones positivas, cada banco fallido contribuye hasta 4 trimestres de señal previa. Tenemos una 
tasa de positivos del 0.05% (103 observaciones positivas sobre 206.1291) lo cual hace que el dataset presente un desbalanceo extremo de clases. Esto es normal en riesgo bancario post-crisis, ya que el sistema bancario americano está muy saneado desde 2016. Esto tiene implicaciones directas para el pipeline:

- `AUC-PR` será la métrica principal, no `AUC-ROC`, ya que con tan pocos positivos el `AUC-ROC` puede ser engañosamente alto.
- Aplicar técnicas de manejo de desbalanceo, _class weights_ en `TabPFN`, _oversampling_ con `SMOTE`, o umbral de clasificación ajustado.

La correlación `punto-biserial` tendrá señal débil pero calculable.

Con todo el estudio realizado y creadas las flags de riesgo, procedemos a realizar la correlación `punto-biserial`. 

In [25]:
from scipy.stats import pointbiserialr

# 1 — Cargar panel temporal completo con el flag
panel_labeled = pd.read_parquet(PROCESSED_DIR / "panel_temporal_labeled.parquet")

# Unir con el panel temporal completo para tener las 409 variables
panel_full = pd.read_parquet(PROCESSED_DIR / "panel_temporal.parquet")
panel_full = panel_full.merge(
    panel_labeled[['CERT', 'period', 'failed']], 
    on=['CERT', 'period'], 
    how='left'
)
panel_full['failed'] = panel_full['failed'].fillna(0).astype(int)

print(f"Panel completo con flag: {panel_full.shape}")
print(f"Positivos: {panel_full['failed'].sum()} | Negativos: {(panel_full['failed']==0).sum()}")

Panel completo con flag: (206129, 412)
Positivos: 103 | Negativos: 206026


In [29]:
import warnings
from scipy.stats import ConstantInputWarning
# 2 — Calcular correlación punto-biserial para cada variable numérica
id_cols = ['CERT', 'period', 'failed']

# Eliminar variables con varianza cero — no informativas
feature_cols = [c for c in panel_full.columns 
                if c not in id_cols 
                and panel_full[c].std() > 0]
print(f"Variables con varianza > 0: {len(feature_cols)}")
print(f"Variables descartadas por varianza cero o casi cero: {len([c for c in panel_full.columns if c not in id_cols]) - len(feature_cols)}")

results = []
for col in feature_cols:
    try:
        mask = panel_full[col].notna()
        if mask.sum() < 100:
            continue
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=ConstantInputWarning)
            corr, pval = pointbiserialr(
                panel_full.loc[mask, 'failed'],
                panel_full.loc[mask, col].astype(float)
        )
        results.append({
            'variable': col,
            'correlation': round(corr, 4),
            'abs_correlation': round(abs(corr), 4),
            'pvalue': round(pval, 6),
        })
    except Exception as e:
        print(f"Error en {col}: {e}")

df_corr = pd.DataFrame(results).sort_values('abs_correlation', ascending=False)
print(f"\nCorrelaciones calculadas: {len(df_corr)}")
print(f"\nTop 20 variables más correlacionadas con failures:")
print(df_corr.head(20).to_string(index=False))

Variables con varianza > 0: 402
Variables descartadas por varianza cero o casi cero: 7

Correlaciones calculadas: 401

Top 20 variables más correlacionadas con failures:
variable  correlation  abs_correlation   pvalue
  RBCPCA       0.3020           0.3020 0.000000
 AVTANEQ       0.1488           0.1488 0.000000
 LNAGT1R       0.1380           0.1380 0.000000
 LNCIT1R       0.1312           0.1312 0.000000
  RBCERI      -0.1197           0.1197 0.000000
LNCONT1R       0.1044           0.1044 0.000000
    ROEQ      -0.0636           0.0636 0.000000
CHBALRCR       0.0508           0.0508 0.000000
     EQ2      -0.0505           0.0505 0.322077
NCRELOCR       0.0492           0.0492 0.000000
     EQ5      -0.0471           0.0471 0.375599
  CHBAL2       0.0399           0.0399 0.050439
 CT1BADJ      -0.0390           0.0390 0.440273
   CHBAL       0.0374           0.0374 0.048936
    CHFL      -0.0362           0.0362 0.000000
   CHFLA      -0.0358           0.0358 0.000000
   P3RER      

In [27]:
# 3 — Enriquecer con info CAMELS y guardar
df_selected = pd.read_csv(project_root / "docs" / "camels_selected.csv")
df_corr = df_corr.merge(
    df_selected[['variable', 'source', 'camels', 'description']], 
    on='variable', 
    how='left'
)

# Guardar
df_corr.to_csv(project_root / "docs" / "camels_failure_correlation.csv", index=False)
print(f"Guardado: camels_failure_correlation.csv")

# Resumen por componente CAMELS con failure
print(f"\nCorrelación media por componente CAMELS:")
print(df_corr.groupby('camels')['abs_correlation'].agg(['mean','max','count'])
      .sort_values('mean', ascending=False))

Guardado: camels_failure_correlation.csv

Correlación media por componente CAMELS:
            mean     max  count
camels                         
C       0.024912  0.3020     51
E       0.009040  0.1197     94
L       0.008998  0.0508     65
A       0.006826  0.0345     62
M       0.005160  0.0199     40
S       0.003267  0.0158     83


`RBCPCA` con correlación 0.302 es la variable más discriminativa, esta variable es básicamente la clasificación regulatoria de capitalización del banco. Los bancos que van a quebrar tienen peor categoría regulatoria de capital trimestres antes.

El componente `C` (`Capital`) tiene la mayor correlación media (0.025) y el mayor máximo (0.302), lo que nos confirma que la adecuación de capital es el predictor más potente de quiebra, exactamente lo que predice la teoría `CAMELS`.

Variables con p-value alto como `EQ2`, `EQ5`, `CT1BADJ`, no son estadísticamente significativas.

El análisis de correlación `punto-biserial` con el evento de quiebra confirma la relevancia del componente `C` (`Capital`) como principal predictor, con `RBCPCA` obteniendo la correlación más alta (r=0.302, p<0.001). El componente `S` (`Sensitivity`) presenta la menor correlación media, consistente con el bajo nivel de actividad en derivados de la banca comunitaria americana en el periodo analizado.

Ahora hacemos un estudio del `p-value`. El `p-value` nos dice si la correlación observada es estadísticamente significativa o podría ser producto del azar. Con 206.129 observaciones casi cualquier correlación será significativa, ya que el test tiene muchísima potencia estadística. Pero hay variables donde hemos calculado la sobre un subconjunto pequeño de observaciones (las que no tienen NaN) y ahí el `p-value` sí discrimina.

Con 103 positivos sobre 206.129 observaciones, una correlación de 0.04 ya sale significativa simplemente por el tamaño muestral. El p-value aquí no es el criterio más útil para seleccionar variables.
Lo que sí tiene sentido hacer es un umbral de correlación mínima — por ejemplo eliminar variables con correlación absoluta <0.01, que son variables que prácticamente no discriminan entre bancos sanos y fallidos independientemente del p-value.

In [30]:
# Ver cuántas variables quedan con distintos umbrales de correlación
for threshold in [0.01, 0.02, 0.03, 0.05]:
    n = (df_corr['abs_correlation'] >= threshold).sum()
    print(f"Correlación >= {threshold}: {n} variables")

Correlación >= 0.01: 98 variables
Correlación >= 0.02: 36 variables
Correlación >= 0.03: 21 variables
Correlación >= 0.05: 9 variables


Con solo 103 positivos en 206k observaciones, el umbral de correlación tiene que ser bajo (menos que el tipico de 0.05) ya que exigir demasiado eliminaría variables que sí son relevantes teóricamente pero cuya señal estadística es débil por el extremo desbalanceo. Viendo los resultados obtenidos un umbral de 0.05 tipico nos deja solo 9 variables (de 409 posibles), lo cual nos haria perder demasiada información que aunque teoricamente no es relevante si que puede serlo en la practica. Por tanto tomamos un umbral de 0.01, lo cual nos deja 98 variables (con correlación ≥0.01) que es un subconjunto manejable y estadísticamente justificable. Además coincide bien con la literatura, ya que con tasas de quiebra tan bajas post-2016, correlaciones de 0.01-0.05 son normales y esperadas, es decir umbrales más estrictos eliminarían variables con relevancia teórica `CAMELS` demostrada pero con señal estadística débil por la baja tasa de quiebras en el periodo 2016-2025.

In [31]:
# Selección final — correlación >= 0.01
final_vars = df_corr[df_corr['abs_correlation'] >= 0.01]['variable'].tolist()
print(f"Variables finales seleccionadas: {len(final_vars)}")

# Ver distribución por componente CAMELS
df_corr_final = df_corr[df_corr['abs_correlation'] >= 0.01].copy()
df_corr_final = df_corr_final.merge(
    df_selected[['variable', 'source', 'camels', 'description']],
    on='variable', how='left'
)

print(f"\nDistribución por componente CAMELS:")
print(df_corr_final.groupby('camels')['abs_correlation']
      .agg(['mean','max','count'])
      .sort_values('mean', ascending=False))

# Guardar selección final
df_corr_final.to_csv(project_root / "docs" / "camels_final_selection.csv", index=False)
print(f"Guardado: camels_final_selection.csv")

Variables finales seleccionadas: 98

Distribución por componente CAMELS:
            mean     max  count
camels                         
C       0.063911  0.3020     19
L       0.023239  0.0508     18
E       0.019967  0.1197     33
A       0.017493  0.0345     15
M       0.013520  0.0199      5
S       0.013350  0.0158      8
Guardado: camels_final_selection.csv


Finalmente tenemos los siguientes documentos de claves `CAMEL` para nuestro pipeline
```p
camels_mapping.csv              ← mapeo automático CAMELS de todas las variables
camels_selected.csv             ← 409 variables tras limpieza y correlación >0.95
camels_failure_correlation.csv  ← correlaciones punto-biserial de las 401 variables
camels_final_selection.csv      ← 98 variables finales seleccionadas para el pipeline
``` 

El proceso de ingenieria de caracteristicas que hemos realizado nos ha llevado de 409 a 98 variables:

```m
3.369 columnas FTS + 1.098 CDI + 254 RAT = 4.721 columnas raw totales
    ↓ limpieza NaN >50% + varianza cero
2.494 columnas limpias
    ↓ mapeo CAMELS — solo variables con asignación teórica
624 variables CAMELS
    ↓ eliminación correlación >0.95 entre variables
409 variables sin redundancia
    ↓ correlación punto-biserial con failures ≥0.01
98 variables finales
```

Aunque con las 409 caracteristicas que teniamos antes de implemetar los `failures` mediante la correlacion `punto-biserial` no colapsaba `TabPFN` (límite 2.000). Tenemos que el numero final de 98 es mejor por las siguientes razones: Con 103 positivos sobre las 206k observaciones, tener menos `features` reduce el riesgo de `overfitting` ya que el modelo no puede apoyarse en variables ruidosas.La metrica de interpretabilidad `SHAP` con 98 variables es mucho más eficaz que con 409, ya que sino el gráfico de importancias sería ilegible y pcoo explicativo.

Este proceso de reducción de caracteristicas en sí es un resultado metodológico ya que se demuestra que de miles de variables regulatorias bancarias, solo 98 tienen poder discriminativo real para predecir quiebras. Finalmete creamos el panel temporal final, que contiene las 98 variables financieras seleccionadas mediante `CAMELS`y `Failures` como criterio de selección adiccional.


In [33]:
# Panel temporal final con las 98 variables seleccionadas
panel_temporal_final = pd.read_parquet(PROCESSED_DIR / "panel_temporal.parquet")

# Filtrar solo las 98 variables finales + identificadores
id_cols = ['CERT', 'period']
cols_to_keep = id_cols + [c for c in final_vars if c in panel_temporal_final.columns]

panel_temporal_final = panel_temporal_final[cols_to_keep]
print(f"Panel temporal final: {panel_temporal_final.shape}")

panel_temporal_final.to_parquet(PROCESSED_DIR / "panel_temporal_camels98.parquet", index=False)
print(f"Guardado: panel_temporal_camels98.parquet")

Panel temporal final: (206129, 100)
Guardado: panel_temporal_camels98.parquet


De esta forma ahora tenemos tres paneles finales que serviran como ingesta de datos procesada para nuestro modelo hibrido de `ML` (todos almacenados en el contenedor de datos del disco duro `D:`)

1. `panel_temporal_camels98.parquet`: (206.129 × 100)  features financieras CAMELS 98 variables + `CERT` + `period` el cual alimenta TabPFN.

2. `panel_temporal_labeled.parquet`: (206.129 × 3)  target supervisado `CERT` + `period` + `failed` → label para entrenamiento y validación.

3. `panel_nodos.parquet `: (219.576 × 23)   features estructurales de identidad, geografía, holding, eventos MERG → define nodos y atributos del grafo dinámico temporal.